# RAG Pipeline — End-to-End Demo

This notebook walks through the full RAG pipeline:
1. Ingest documents into FAISS vector store
2. Query the retriever directly
3. Run the full RAG chain with LLM generation
4. Inspect retrieval quality and relevance scores

**Run `ingest.py` first** if you haven't already:
```bash
python src/ingest.py --docs_path data/sample_docs/
```

In [ ]:
import sys
sys.path.append('../src')

from ingest import run_ingestion, load_config
from retriever import Retriever
from rag_chain import RAGChain
from local_llm import OllamaLLM
from pathlib import Path

cfg = load_config()
print('Config loaded:')
print(f"  Embedding model : {cfg['embedding']['model']}")
print(f"  Chunk size      : {cfg['chunking']['chunk_size']}")
print(f"  Top-k retrieval : {cfg['retrieval']['top_k']}")
print(f"  LLM model       : {cfg.get('llm', {}).get('model', 'mistral')}")

In [ ]:
# Step 1 — Ingest documents (skip if already done)
index_path = Path('../data/index/faiss.index')

if not index_path.exists():
    print('Building index...')
    run_ingestion(docs_path=Path('../data/sample_docs/'))
else:
    print(f'Index already exists at {index_path}')

In [ ]:
# Step 2 — Test the retriever directly
retriever = Retriever()

test_queries = [
    'What is State of Health in batteries?',
    'thermal simulation guidelines',
    'SOC estimation Kalman filter',
]

for query in test_queries:
    results = retriever.retrieve(query, top_k=3)
    print(f"\nQuery: '{query}'")
    for i, r in enumerate(results, 1):
        print(f"  [{i}] score={r['score']:.4f} | {Path(r['source']).name}")
        print(f"      {r['text'][:120]}...")

In [ ]:
# Step 3 — Check LLM availability
llm = OllamaLLM(model=cfg.get('llm', {}).get('model', 'mistral'))
print(f'LLM status  : {llm}')
print(f'Available models: {llm.list_models()}')

In [ ]:
# Step 4 — Run the full RAG chain
chain = RAGChain()

question = 'What is State of Health (SOH) in batteries?'
result   = chain.query(question, retrieval_only=not llm.is_available())

print(chain.format_response(result))

In [ ]:
# Step 5 — Visualise retrieval relevance scores
import matplotlib.pyplot as plt
import numpy as np

queries = [
    'battery SOH degradation',
    'thermal management cooling',
    'mesh density simulation',
    'cell balancing strategy',
]

fig, axes = plt.subplots(1, len(queries), figsize=(14, 4))

for ax, query in zip(axes, queries):
    results = retriever.retrieve(query, top_k=5)
    scores  = [r['score'] for r in results]
    labels  = [f"Chunk {r['chunk_id']}" for r in results]
    
    ax.barh(labels, scores, color='#2563EB', alpha=0.8)
    ax.set_xlim(0, 1)
    ax.set_xlabel('Cosine similarity')
    ax.set_title(query[:25] + '...', fontsize=9)
    ax.grid(True, axis='x', alpha=0.3)

plt.suptitle('Retrieval Relevance Scores per Query', fontsize=12)
plt.tight_layout()
plt.show()